# Clase 127 — TensorFlow Datasets (TFDS)

**TFDS** es un catálogo de datasets prearmados (CIFAR, MNIST, IMDB, COCO...) con
descarga y cache automáticos. Vemos `tfds.load(..., split=, as_supervised=True,
with_info=True)` y la alternativa moderna Hugging Face `datasets`.

Requiere: `tensorflow-datasets` (`pip install tensorflow-datasets`). **`tfds.load`
descarga a `~/tensorflow_datasets` la primera vez**; este notebook es de referencia
y no se ejecuta aquí.

## 1. Importar TFDS y listar el catálogo

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
AUTOTUNE = tf.data.AUTOTUNE

# tfds.load descarga y cachea en ~/tensorflow_datasets la primera vez.
print("datasets disponibles (primeros 10):", tfds.list_builders()[:10])

## 2. Cargar CIFAR-10 con `as_supervised` y `with_info`

In [ ]:
(ds_train, ds_test), info = tfds.load(
    "cifar10",
    split=["train", "test"],
    as_supervised=True,          # devuelve tuplas (imagen, etiqueta)
    with_info=True,
)
print("clases:", info.features["label"].num_classes)
print("ejemplos train:", info.splits["train"].num_examples)
print("shape imagen:", info.features["image"].shape)

## 3. Splits con la slicing API

In [ ]:
ds_tr, ds_val = tfds.load(
    "cifar10",
    split=["train[:90%]", "train[90%:]"],   # 90/10 train/val
    as_supervised=True,
)
print("train 90%:", int(ds_tr.cardinality()))
print("val   10%:", int(ds_val.cardinality()))

## 4. Pipeline de entrenamiento

In [ ]:
def preprocesar(imagen, etiqueta):
    imagen = tf.cast(imagen, tf.float32) / 255.0
    imagen = tf.reshape(imagen, (32 * 32 * 3,))
    return imagen, etiqueta

train = (ds_train.map(preprocesar, num_parallel_calls=AUTOTUNE)
         .cache().shuffle(1024).batch(128).prefetch(AUTOTUNE))
test = (ds_test.map(preprocesar, num_parallel_calls=AUTOTUNE)
        .batch(128).prefetch(AUTOTUNE))
print("pipeline listo:", train.element_spec)

## 5. MLP sobre CIFAR-10

In [ ]:
modelo = keras.Sequential([
    keras.Input((32 * 32 * 3,)),
    layers.Dense(1024, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(train, validation_data=test, epochs=10, verbose=2)

## 6. Alternativa moderna: Hugging Face `datasets`

In [ ]:
# Estándar multi-framework en NLP/LLMs (basado en Arrow):
from datasets import load_dataset

imdb = load_dataset("imdb")                 # descarga desde el Hub
print(imdb)
# convertir a tf.data para entrenar con Keras:
tf_ds = imdb["train"].to_tf_dataset(
    columns="text", label_cols="label", batch_size=32, shuffle=True)
print("HF -> tf.data:", tf_ds.element_spec)

## Ejercicios

1. **Listar**: imprimí los primeros 20 datasets con `tfds.list_builders()`.
2. **CIFAR-10**: cargá con `as_supervised=True, with_info=True` e imprimí el
   `info` (clases, num_examples, shape).
3. **Slicing**: cargá `train[:90%]` y `train[90%:]` como train/val.
4. **Pipeline**: `map(preprocess).cache().shuffle(1024).batch(32).prefetch(AUTOTUNE)`.
5. **HF datasets**: `load_dataset('imdb')`, inspeccionalo y convertilo con
   `to_tf_dataset(...)`.

## Conclusiones

- `tfds.load(nombre, split=, as_supervised=True, with_info=True)` descarga, cachea y devuelve `(ds, info)`.
- `as_supervised=True` da tuplas `(x, y)`; sin él, un dict con todas las features.
- La **slicing API** (`'train[:90%]'`, `'train[90%:]'`) hace splits reproducibles.
- `info` trae metadata (num_classes, shape, splits) útil para armar el modelo.
- Para NLP/LLMs modernos, Hugging Face `datasets` es el estándar; `to_tf_dataset` lo conecta a Keras.